# 1. Load train_cf

In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix

from sklearn.neighbors import NearestNeighbors

train_cf = pd.read_parquet(
    "../data/train_cf.parquet"
)

test = pd.read_parquet(
    "../data/test.parquet"
)

# 2. Encode users/items

In [2]:
user_ids = train_cf["reviewerID"].unique()
item_ids = train_cf["asin"].unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(user_ids)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(item_ids)
}

# 3. Build sparse matrix

In [3]:
rows = train_cf["reviewerID"].map(
    user_to_idx
)

cols = train_cf["asin"].map(
    item_to_idx
)

data = np.ones(len(train_cf))

interaction_matrix = csr_matrix(
    (
        data,
        (rows, cols)
    ),
    shape=(
        len(user_ids),
        len(item_ids)
    )
)

print("Interaction Matrix Shape: ", interaction_matrix.shape)

Interaction Matrix Shape:  (190963, 62707)


# 4. Train Item KNN

In [4]:
knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21
)

knn.fit(
    interaction_matrix.T
)


idx_to_item = {
    v: k
    for k, v in item_to_idx.items()
}

# Similar Products Function

In [5]:
def get_similar_items(
    asin,
    n=10
):
    
    if asin not in item_to_idx:
        return []
    
    item_idx = item_to_idx[asin]

    distances, indices = knn.kneighbors(
        interaction_matrix.T[item_idx],
        n_neighbors=n + 1
    )

    neighbors = []

    for idx in indices.flatten()[1:]:
        neighbors.append(
            idx_to_item[idx]
        )

    return neighbors

In [6]:
sample_item = train_cf["asin"].iloc[0]

get_similar_items(
    sample_item,
    n=10
)

['B000Q01YUO',
 'B002KGLDXU',
 'B00AFV237W',
 'B00GSPUES4',
 'B0014SWPX2',
 'B0064NTUPI',
 'B00BJ7DQMC',
 'B00358PJ28',
 'B0075790SO',
 'B004MNUTDY']

## User-Level Recommendation Generation

Build User History Lookup

In [7]:
user_history = (
    train_cf.groupby("reviewerID")["asin"]
    .apply(set)
    .to_dict()
)

Recommendation Function

In [8]:
def recommend_for_user(
    user_id,
    top_k=10
):

    if user_id not in user_history:
        return []

    interacted_items = user_history[user_id]

    candidate_scores = {}

    for item in interacted_items:

        similar_items = get_similar_items(
            item,
            n=20
        )

        for sim_item in similar_items:

            if sim_item in interacted_items:
                continue

            candidate_scores[sim_item] = (
                candidate_scores.get(sim_item, 0) + 1
            )

    ranked_items = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = [
        item
        for item, score in ranked_items[:top_k]
    ]

    return recommendations

Test

In [9]:
sample_user = train_cf["reviewerID"].iloc[0]

recommend_for_user(
    sample_user,
    top_k=10
)

['B00AFV237W',
 'B00GSPUES4',
 'B000Q01YUO',
 'B002KGLDXU',
 'B0014SWPX2',
 'B0064NTUPI',
 'B004MNUTDY',
 'B00BJ7DQMC',
 'B0075790SO',
 'B00358PJ28']

In [10]:
test = pd.read_parquet("../data/test.parquet")

ground_truth = (
    test.groupby("reviewerID")["asin"]
        .apply(set)
        .to_dict()
)

In [11]:
def precision_at_k(
    recommended,
    relevant,
    k=10
):
    recommended = recommended[:k]

    hits = len(
        set(recommended)
        &
        set(relevant)
    )

    return hits / k

In [12]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def evaluate_item_cf(
    sample_size=5000,
    seed=42
):

    np.random.seed(seed)

    users = np.array(
        list(ground_truth.keys())
    )

    sample_users = np.random.choice(
        users,
        size=sample_size,
        replace=False
    )

    precisions = []

    for user in tqdm(
        sample_users,
        desc=f"Seed {seed}"
    ):

        recommendations = recommend_for_user(
            user,
            top_k=10
        )

        relevant_items = ground_truth[user]

        p = precision_at_k(
            recommendations,
            relevant_items,
            k=10
        )

        precisions.append(p)

    return np.mean(precisions)

In [14]:
seeds = [42, 123, 999]

results = []

for seed in seeds:

    precision = evaluate_item_cf(
        sample_size=1000,
        seed=seed
    )

    results.append(precision)

    print(
        f"Seed {seed}: "
        f"{precision:.6f}"
    )

Seed 42:   0%|          | 1/1000 [00:00<03:30,  4.75it/s]

Seed 42: 100%|██████████| 1000/1000 [02:17<00:00,  7.26it/s]


Seed 42: 0.000900


Seed 123: 100%|██████████| 1000/1000 [02:21<00:00,  7.06it/s]


Seed 123: 0.001500


Seed 999: 100%|██████████| 1000/1000 [02:17<00:00,  7.25it/s]

Seed 999: 0.000800
